In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import Lasso, ElasticNet, LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.base import clone
from catboost import CatBoostRegressor
from celer import FusedLasso

In [12]:
# Helper Functions

def make_group_design(n, rng, n_groups):
    """
    One-hot encoding of group instances.
    Each observation belongs to exactly one group.
    """
    group_id = rng.integers(0, n_groups, size=n)
    X = np.zeros((n, n_groups))
    X[np.arange(n), group_id] = 1.0
    return X, group_id


def make_group_sparse_beta(n_groups, s_unique, rng):
    """
    s_unique = number of unique coefficient values
    group sparsity = n_groups >> s_unique
    """
    unique_vals = rng.uniform(0.5, 1.5, size=s_unique)
    beta = np.resize(unique_vals, n_groups)
    return beta

In [ ]:
# --- Hyperparameter grids ---
LASSO_GRID = {"alpha": np.logspace(-3, 0, 8)}
EN_GRID = {"alpha": np.logspace(-3, 0, 8), "l1_ratio": [0.2, 0.5, 0.8]}
RF_GRID = {"max_depth": [3, 5, None], "min_samples_leaf": [5, 10, 20]}
GB_GRID = {"learning_rate": [0.01, 0.05, 0.1], "max_depth": [2, 3]}
CATBOOST_GRID = {"learning_rate": [0.01, 0.05, 0.1], "depth": [3, 5, 7],"iterations": [100, 300]}
FUSEDLASSO_GRID = {"alpha": [0.01, 0.05, 0.1], "l1_ratio": [0.2, 0.5, 0.8]}

# --- Tune a single learner ---
def tune_learner(model, param_grid, X, y):
    if param_grid is None:
        model.fit(X, y)
        return model
    gs = GridSearchCV(model, param_grid, cv=3, scoring="neg_mean_squared_error", n_jobs=1)
    gs.fit(X, y)
    return gs.best_estimator_

# --- Tune all learners once for a DGP ---
def tune_once(dgp_func, learners, n, seed=0):
    rng = np.random.default_rng(seed)
    X, D, Y, _ = dgp_func(n, rng)
    tuned = {}
    for name, model in learners.items():
        if name == "Lasso":
            grid = LASSO_GRID
        elif name == "ElasticNet":
            grid = EN_GRID
        elif name == "RF":
            grid = RF_GRID
        elif name == "GB":
            grid = GB_GRID
        elif name == "CatBoost":
            grid = CATBOOST_GRID
        elif name == "FusedLasso":
            grid = FUSEDLASSO_GRID
        else:
            tuned[name] = clone(model)
            continue
    tuned[name] = tune_learner(clone(model), grid, X, Y)

In [14]:
def dgp1(n, rng):
    n_groups = 6       # e.g. (Low/Mid/High) × (Male/Female)
    s_unique = 6       # no sparsity

    X, _ = make_group_design(n, rng, n_groups)

    beta0 = rng.uniform(0.5, 1.5, size=n_groups)
    beta1 = beta0 + 1.0

    eps0 = rng.normal(0, 1, size=n)
    eps1 = rng.normal(0, 1, size=n)

    Y0 = X @ beta0 + eps0
    Y1 = X @ beta1 + eps1

    D = rng.binomial(1, 0.5, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [15]:
def dgp2(n, rng):
    n_groups = int(0.1 * n)
    s_unique = 5       # strong group sparsity

    X, _ = make_group_design(n, rng, n_groups)

    beta0 = make_group_sparse_beta(n_groups, s_unique, rng)
    beta1 = beta0 + 1.0

    eps0 = rng.normal(0, 1, size=n)
    eps1 = rng.normal(0, 1, size=n)

    Y0 = X @ beta0 + eps0
    Y1 = X @ beta1 + eps1

    D = rng.binomial(1, 0.5, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [16]:
def dgp3(n, rng):
    n_groups = int(1.7 * n)
    s_unique = n_groups  # no sparsity at all

    X, _ = make_group_design(n, rng, n_groups)

    beta0 = rng.uniform(0.5, 1.5, size=n_groups)
    beta1 = beta0 + 1.0

    eps0 = rng.normal(0, 1, size=n)
    eps1 = rng.normal(0, 1, size=n)

    Y0 = X @ beta0 + eps0
    Y1 = X @ beta1 + eps1

    D = rng.binomial(1, 0.5, size=n)
    Y = D * Y1 + (1 - D) * Y0

    return X, D, Y, 1.0

In [17]:
def cross_fit_nuisances_fast(X, D, Y, learner, K=2):
    n = X.shape[0]
    m0_hat = np.zeros(n)
    m1_hat = np.zeros(n)
    e_hat = np.zeros(n)
    prop_model = LogisticRegression(max_iter=1000)
    kf = KFold(n_splits=K, shuffle=True, random_state=123)

    for train_idx, test_idx in kf.split(X):
        X_tr, X_te = X[train_idx], X[test_idx]
        D_tr = D[train_idx]
        Y_tr = Y[train_idx]

        # Propensity model
        prop = clone(prop_model)
        prop.fit(X_tr, D_tr)
        e_hat[test_idx] = prop.predict_proba(X_te)[:, 1]

        # Outcome models (already tuned)
        m1 = clone(learner)
        m0 = clone(learner)
        m1.fit(X_tr[D_tr == 1], Y_tr[D_tr == 1])
        m0.fit(X_tr[D_tr == 0], Y_tr[D_tr == 0])

        m1_hat[test_idx] = m1.predict(X_te)
        m0_hat[test_idx] = m0.predict(X_te)

    return m0_hat, m1_hat, e_hat

In [18]:
def aipw(Y, D, m0_hat, m1_hat, e_hat, tau_true):
    e_hat = np.clip(e_hat, 0.05, 0.95)

    psi = (
        D * (Y - m1_hat) / e_hat
        - (1 - D) * (Y - m0_hat) / (1 - e_hat)
        + m1_hat - m0_hat
    )

    tau_hat = psi.mean()
    var_hat = np.mean((psi - tau_hat) ** 2)
    se = np.sqrt(var_hat / len(Y))

    covered = (tau_hat - 1.96 * se <= tau_true <= tau_hat + 1.96 * se)

    return tau_hat, se, covered, e_hat

In [19]:
def monte_carlo(dgp, learners, n, sims=200):
    rows = []

    for s in range(sims):
        rng = np.random.default_rng(s)
        X, D, Y, tau_true = dgp(n, rng)

        for name, learner in learners.items():
            m0, m1, e = cross_fit_nuisances_fast(X, D, Y, learner)
            tau, se, cov, e_hat = aipw(Y, D, m0, m1, e, tau_true)

            rows.append({
                "Learner": name,
                "tau": tau,
                "se": se,
                "covered": cov,
                "e_mean": e_hat.mean(),
                "e_var": e_hat.var()
            })

    df = pd.DataFrame(rows)

    return df.groupby("Learner").agg(
        Mean=("tau", "mean"),
        Bias=("tau", lambda x: x.mean() - tau_true),
        RMSE=("tau", lambda x: np.sqrt(np.mean((x - tau_true)**2))),
        Coverage=("covered", "mean"),
        Mean_SE=("se", "mean"),
        PropensityVar=("e_var", "mean")
    )

In [ ]:
learners_regime = {
    "OLS": LinearRegression(),
    "Lasso": Lasso(max_iter=5000),
    "ElasticNet": ElasticNet(max_iter=5000),
    "RF": RandomForestRegressor(n_estimators=300, random_state=123, n_jobs=1),
    "GB": GradientBoostingRegressor(n_estimators=300, random_state=123),
    "CatBoost": CatBoostRegressor(verbose=0, random_state=123, iterations=300),
    "FusedLasso": FusedLasso(alpha=0.05, l1_ratio=0.5, max_iter=1000)
}

In [21]:
# DGP 1
print("DGP 1")
tuned_learners_1 = tune_once(dgp1, learners_regime, n=100)
print(monte_carlo(dgp1, tuned_learners_1, n=100, sims=100))

# DGP 2
print("DGP 2")
tuned_learners_2 = tune_once(dgp2, learners_regime, n=100)
print(monte_carlo(dgp2, tuned_learners_2, n=100, sims=100))

# DGP 3
print("DGP 3")
tuned_learners_3 = tune_once(dgp3, learners_regime, n=100)
print(monte_carlo(dgp3, tuned_learners_3, n=100, sims=100))

DGP 1
                Mean      Bias      RMSE  Coverage   Mean_SE  PropensityVar
Learner                                                                    
ElasticNet  1.003297  0.003297  0.263695      0.93  0.237492       0.013455
GB          1.004689  0.004689  0.288436      0.93  0.243625       0.013455
Lasso       1.006498  0.006498  0.277980      0.93  0.241018       0.013455
OLS         1.002894  0.002894  0.302230      0.93  0.247781       0.013455
RF          1.003119  0.003119  0.237739      0.94  0.236116       0.013455
DGP 2
                Mean      Bias      RMSE  Coverage   Mean_SE  PropensityVar
Learner                                                                    
ElasticNet  1.000737  0.000737  0.221761      0.96  0.237653       0.015815
GB          1.010603  0.010603  0.238167      0.96  0.248110       0.015815
Lasso       1.000284  0.000284  0.221824      0.96  0.237693       0.015815
OLS         1.011151  0.011151  0.250965      0.96  0.258508       0.015815
